# P4 프로젝트: AI 문서 검색 및 근거 기반 질의응답

**핵심 질문:** *단어가 정확히 일치하지 않아도 관련 문서를 찾을 수 있을까? 그리고 찾은 문서만 근거로 LLM이 답변하도록 할 수 있을까?*

이 프로젝트는 **Encoder-only 계열 모델의 문장 임베딩 → 문서 검색 → Decoder-only LLM의 답변 생성**을 연결한다. 검색과 생성이 결합된 간단한 **RAG(Retrieval-Augmented Generation, 검색 증강 생성)** 사례다.

**중요:** 모델의 가중치를 **학습하거나 변경하지 않는다.** 사전훈련된 모델을 사용하여 직접 실험하고 결과를 해석한다.

**학습 자료:** 함께 제공된 `P4_project_documents_25.json`에는 **강의용으로 직접 작성한 가상 '한솔 생태관측소' 문서 25개**가 있다. 실제 관측소의 운영 규정이나 기관 자료가 아니다. 모델의 근거 문서 역할만 한다.

**준비물:** 이 노트북과 JSON 파일을 **같은 폴더**에 두고 실행한다. VS Code/Jupyter 또는 Google Colab 사용 가능. 첫 실행에는 인터넷이 필요하다.


## 학습 목표와 제출물

1. **Encoder-only 계열 임베딩 모델**로 질문·문서의 의미를 숫자 벡터로 나타내고 관련 문서를 검색한다.
2. 표면적인 글자 일치 검색과 **의미 기반 검색**을 비교한다.
3. **Decoder-only 언어 모델**에 검색 문서를 제공하여 근거 기반 답변을 생성한다.
4. 답변이 정말 문서에 근거하는지 확인한다. 문서에 없는 질문에는 **모른다고 답하는지** 평가한다.

**제출:** (1) 질문 5개 이상으로 수행한 검색 비교표, (2) 좋은 답변과 잘못된 답변 사례 각 1개, (3) 근거 문서의 정확성 점검, (4) 실험 소감과 개선 제안 5~10문장.

**AI 활용:** AI에게 코드 의미나 오류 원인을 질문할 수 있다. 단, **실제 실행 결과와 AI가 예상한 결과를 구분**하여 기록한다.


---
## 0. 설치와 실행 환경

* 최초 한 번만 패키지를 설치한다.
* CPU에서도 실행할 수 있지만 **문장 생성은 느릴 수 있다**. GPU가 있으면 권장한다.
* 사용하는 사전훈련 모델: 문장 임베딩은 `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, 답변 생성은 `Qwen/Qwen2.5-0.5B-Instruct`.
* 두 모델의 파일을 최초에 다운로드하므로 네트워크 환경과 여유 저장 공간이 필요하다.
* 모델을 직접 학습하거나 파인튜닝하는 코드는 없다.


In [ ]:
%pip install -q "sentence-transformers>=3,<6" "transformers>=4.47,<5" scikit-learn pandas matplotlib

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import torch
from IPython.display import display

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# 처음에는 검색만 끝까지 실행하고, 아래 '답변 생성' 절에서 LLM을 불러온다.
TOP_K = 3
print("PyTorch:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
print("작업 폴더:", Path.cwd())

---
## 1. 교육용 문서 25개 불러오기

과제 시작을 위해 웹페이지 25개를 직접 수집할 필요는 없다. **미리 준비된 가상 문서 25개**로 검색과 답변 생성 과정을 먼저 실험한다.

문서에는 `id`(문서 번호), `category`(주제), `title`(제목), `text`(본문)가 있다. 나중에 학생이 선택한 실제 자료로 교체할 수 있다.

> **Colab 사용 시:** 왼쪽 파일 패널에 `P4_project_documents_25.json`을 업로드한 후 아래 셀을 실행한다.


In [ ]:
DATA_FILE = Path("P4_project_documents_25.json")
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"{DATA_FILE}을(를) 찾지 못했습니다. 현재 폴더: {Path.cwd()}\n"
        "노트북과 JSON 파일을 같은 폴더에 두거나 Colab에 업로드하세요."
    )

docs = json.loads(DATA_FILE.read_text(encoding="utf-8"))
assert len(docs) > 0, "문서는 한 개 이상 필요합니다."
if not (20 <= len(docs) <= 30):
    print("참고: 학생 프로젝트는 문서 20~30개를 권장합니다.")
assert all(all(key in d for key in ("id", "title", "text")) for d in docs)

print("문서 수:", len(docs))
display(pd.DataFrame(docs)[["id", "category", "title"]])
print("\n문서 한 건 예시:\n", docs[0]["text"])

### 실제 자료로 바꾸고 싶다면: 문서 수집처

기본 실습에는 **첨부된 25개 가상 문서**를 사용하므로 별도 수집이 필요 없다. 이후 학생이 관심 분야의 실제 문서로 교체할 수 있다.

- **국립생태원**의 교육·생태정보 자료: [국립생태원 홈페이지](https://www.nie.re.kr/nie/main/main.do) — 자료별 공개 범위와 재배포 조건 확인
- **국립공원 관련 자료:** [국립공원 생태자원 소개](https://www.korea.kr/news/policyNewsView.do?newsId=148933526) — 공개 내용에서 관련 주제 발췌, 이용 조건 확인
- **위키백과:** 관심 주제의 문서 20~30개를 고른 후 일부를 이용할 수 있으나, **저작자 표시 및 동일조건변경허락(CC BY-SA) 등 라이선스 준수**가 필요함
- **학생이 직접 작성한 짧은 안내문:** 자료 출처가 명확하고 과제 목적에 맞다면 가장 간단함

문서 한 건은 `id`, `title`, `text`가 반드시 있어야 한다. `category`, `source`도 함께 저장할 것을 권한다. 한 문서가 너무 길어지면 검색 범위가 혼합되므로 **한 주제당 3~5문장**으로 시작한다.

**교수자 권장:** 첫 수업에서는 제공된 25개로 실행을 끝낸 후, 확장 과제에서만 실제 자료를 교체한다.


**질문:** 같은 동물을 여러 번 촬영했을 때 **탐지된 객체 수**와 **실제 개체 수**가 다른 이유는 무엇일까? 어떤 문서가 그 질문의 답을 담고 있을까? 문서 목록에서 먼저 예상해 보자.

---
## 2. 문자열 기반 검색과 의미 기반 검색

**문자열 기반 검색**은 질문과 문서에서 겹치는 글자 조각을 찾는다. 여기서는 한국어 형태소 분석 없이도 실행할 수 있는 **문자 n-그램 TF-IDF**를 사용한다. 이는 엄밀한 의미 이해가 아니라 글자 형태의 유사도를 이용하는 비교 기준이다.

**의미 기반 검색**은 사전훈련된 **Encoder-only 계열의 문장 임베딩 모델**을 활용한다. 질문과 문서를 각각 벡터로 변환한 뒤 **코사인 유사도**를 계산한다. 값이 높을수록 비슷한 표현이라고 모델이 판단했다는 뜻이지, 정답임을 보장하는 확률은 아니다.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# 비교 기준: 표면 문자열(제목+본문)의 유사도
corpus = [f"{d['title']} {d['text']}" for d in docs]
lexical_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 3))
lexical_matrix = lexical_vectorizer.fit_transform(corpus)

# 의미 검색: 사전훈련된 Encoder 계열 임베딩 모델 (훈련하지 않음)
EMBED_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMBED_MODEL_ID, device="cpu")
doc_embeddings = embedder.encode(corpus, normalize_embeddings=True, show_progress_bar=False)

print("문서 임베딩 형태:", doc_embeddings.shape)
print("첫 문서 벡터의 앞부분:", np.round(doc_embeddings[0, :8], 3))

In [ ]:
def search_documents(question, method="semantic", top_k=3):
    """질문과 관련된 문서를 점수 순서로 반환한다."""
    if method == "semantic":
        q = embedder.encode([question], normalize_embeddings=True, show_progress_bar=False)
        scores = np.asarray(doc_embeddings @ q[0], dtype=float)
    elif method == "lexical":
        q = lexical_vectorizer.transform([question])
        scores = cosine_similarity(q, lexical_matrix).ravel()
    else:
        raise ValueError("method는 'semantic' 또는 'lexical'이어야 합니다.")

    order = np.argsort(-scores)[:top_k]
    return [
        {"id": docs[i]["id"], "title": docs[i]["title"],
         "text": docs[i]["text"], "score": float(scores[i])}
        for i in order
    ]


def show_results(results):
    display(pd.DataFrame([
        {"순위": rank, "문서 ID": d["id"], "제목": d["title"], "유사도": round(d["score"], 3)}
        for rank, d in enumerate(results, 1)
    ]))

question = "야생동물을 한 번만 세려면 어떻게 해야 하나요?"
print("질문:", question)
print("\n[문자열 기반 검색]")
show_results(search_documents(question, "lexical", TOP_K))
print("\n[의미 기반 검색]")
show_results(search_documents(question, "semantic", TOP_K))

### 실험 1 — 질문 표현을 바꾸면 검색 결과도 바뀔까?

아래 질문들은 강의용 문서의 특정 내용을 찾도록 작성했다. **기준 문서 ID는 교수자가 미리 지정한 간단한 평가 정답**이며, 실제로는 여러 문서가 답에 도움을 줄 수도 있다.

* 같은 질문을 서로 다른 표현으로 바꿔 보고 상위 3개 문서를 비교하자.
* 결과가 어색한 질문도 **수정하거나 삭제하지 말고** 분석해 보자.


In [ ]:
test_questions = [
    {"question": "밤에 동물 사진은 어떤 방식으로 얻나요?", "target_id": "D02"},
    {"question": "야생동물을 한 번만 세려면 어떻게 해야 하나요?", "target_id": "D10"},
    {"question": "동물이 없는데 있다고 표시한 오류를 뭐라고 하나요?", "target_id": "D13"},
    {"question": "동물이 차도를 안전하게 건너도록 만든 시설은?", "target_id": "D16"},
    {"question": "카메라 전원과 용량은 언제 확인하나요?", "target_id": "D05"},
    {"question": "촬영한 장면에 사람 얼굴이 나오면 어떻게 하나요?", "target_id": "D08"},
    {"question": "발견한 동물 종류가 확실하지 않을 때 시민은 어떻게 작성하나요?", "target_id": "D21"},
    {"question": "동물을 유인하려고 간식을 놓아도 되나요?", "target_id": "D20"},
]

rows = []
for item in test_questions:
    q, gold = item["question"], item["target_id"]
    a = search_documents(q, "lexical", TOP_K)
    b = search_documents(q, "semantic", TOP_K)
    rows.append({
        "질문": q,
        "기준 문서": gold,
        "문자열 Top-3": ", ".join(d["id"] for d in a),
        "의미 Top-3": ", ".join(d["id"] for d in b),
        "문자열 Hit@3": int(gold in [d["id"] for d in a]),
        "의미 Hit@3": int(gold in [d["id"] for d in b]),
    })

comparison = pd.DataFrame(rows)
display(comparison)
print("문자열 검색 Hit@3 평균:", round(comparison["문자열 Hit@3"].mean(), 3))
print("의미 검색 Hit@3 평균:", round(comparison["의미 Hit@3"].mean(), 3))
print("* Hit@3: 미리 지정한 기준 문서가 검색 상위 3개에 포함되면 1, 아니면 0")

**해석 질문**

- 문자열 검색보다 의미 검색이 잘된 질문은 무엇인가? 두 검색이 비슷한 결과를 내는 경우는?
- `Hit@3` 평균이 더 높은 방법이 모든 질문에서 더 좋은 것은 아니다. 실제 상위 3개 문서의 관련성도 확인해야 하는 이유는?
- 검색 유사도 점수 `0.8`을 **정답일 확률 80%**라고 해석하면 안 되는 이유는?


---
## 3. Decoder-only 모델로 검색된 문서에 근거하여 답하기

앞 절에서 찾은 **상위 3개 문서의 본문**을 질문과 함께 Decoder-only 언어 모델에 전달한다.

**핵심 구분**

- **Encoder 계열 임베딩 모델:** 문서를 **찾는 역할**
- **Decoder-only 언어 모델:** 찾은 문서를 **읽고 답변을 생성하는 역할**
- **문서가 없는 질문:** 모른다고 답하는 것이 바람직하다. 다만 작은 모델은 지시를 어길 수 있으므로 검증이 필요하다.

이번 프로젝트에서 만드는 것은 외부 검색 도구가 아니라 **25개 문서에 한정된 작은 RAG 시스템**이다.

다운로드·메모리 부담을 고려하여 0.5B(약 5억 파라미터)급 모델을 사용한다. **한국어 답변이 어색하거나, 근거 없이 내용을 만들어낼 수 있다.** 그런 오류 자체도 분석 대상이다.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

GEN_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

llm_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
llm.to(DEVICE)
llm.eval()
print("답변 생성 모델:", GEN_MODEL_ID)
print("실행 장치:", DEVICE)

In [ ]:
def generate_answer(question, retrieved, max_new_tokens=100):
    """검색된 문서만 주고 답변을 생성한다. 모델 가중치는 바뀌지 않는다."""
    context = "\n\n".join(
        f"[{d['id']}] {d['title']}\n{d['text']}" for d in retrieved
    )

    messages = [
        {"role": "system", "content": (
            "너는 수업용 문서 검색 도우미다. 제공된 참고 문서만 근거로 "
            "한국어 두세 문장으로 답해라. 관련 내용이 문서에 없으면 "
            "'제공된 문서에서는 확인할 수 없습니다.'라고 답해라. "
            "문서에 근거가 있다면 관련 문서 번호를 [D03]처럼 표시해라. "
            "문서에 없는 사실이나 문서 번호를 만들어내지 마라."
        )},
        {"role": "user", "content": f"질문: {question}\n\n참고 문서:\n{context}"},
    ]
    input_ids = llm_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(DEVICE)
    with torch.inference_mode():
        generated = llm.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,     # 재현 가능한 Greedy Decoding
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    answer_ids = generated[0][input_ids.shape[-1]:]
    return llm_tokenizer.decode(answer_ids, skip_special_tokens=True).strip()


def ask(question, top_k=3):
    found = search_documents(question, method="semantic", top_k=top_k)
    print("질문:", question)
    print("\n[검색된 근거 문서]")
    show_results(found)
    for d in found:
        print(f"  [{d['id']}] {d['title']}: {d['text']}")
    answer = generate_answer(question, found)
    print("\n[Decoder-only 생성 답변]")
    print(answer)
    cited = re.findall(r"\[D\d{2}\]", answer)
    supplied = {f"[{d['id']}]" for d in found}
    print("\n표시한 문서 ID가 실제 제공된 문서에 있는가?:", all(x in supplied for x in cited) if cited else "인용 표시 없음")
    return {"question": question, "retrieved": found, "answer": answer}

result1 = ask("촬영 장면에 사람 얼굴이 보이면 어떻게 해야 하나요?")

### 실험 2 — 근거가 없는 질문에도 답할까?

다음 질문은 **자료에 없는 내용**을 묻는다. 좋은 답변은 확실하지 않은 내용을 단정하지 않고, 자료에서 확인할 수 없다고 밝히는 것이다.

* 작은 언어 모델은 문서에 없는 내용을 지어낼 수 있다.
* `[Dxx]` 인용 표시가 있어도 **실제 해당 문서에 내용이 있는지 사람이 확인**해야 한다.


In [ ]:
result2 = ask("한솔 생태관측소의 2027년 예산 총액은 얼마인가요?")

print("\n[학생이 확인할 내용]")
print("1. 위 답변에 문서로 확인할 수 없는 숫자가 등장했나요?")
print("2. 근거 문서가 없으면 모른다고 답했나요?")
print("3. 제시된 [Dxx]가 실제로 그 주장을 뒷받침하나요?")

---
## 4. 같은 질문에서 검색을 생략하면 어떻게 될까? (비교)

검색된 근거를 주는 경우와 **문서 없이 답변을 요구**하는 경우를 비교한다. 여기서는 한솔 생태관측소라는 가상 기관의 구체적 운영 기준을 묻는다. 모델이 원래 알 수 없는 내용이다.

> **주의:** 문서를 주어도 모델이 꼭 올바르게 답하는 것은 아니다. 이를 확인하는 것이 프로젝트의 목표다.


In [ ]:
# 근거 문서 없이 질문만 전달한다.
# 같은 Decoder-only 모델을 사용하지만 retrieved=[] 로 문서 제공을 생략한다.
compare_q = "한솔 생태관측소의 장비 점검은 언제 하나요?"

with_docs = search_documents(compare_q, "semantic", top_k=TOP_K)
print("[근거 문서를 제공한 답변]")
print(generate_answer(compare_q, with_docs))
print("\n[근거 문서가 없는 답변]")
print(generate_answer(compare_q, []))

---
## 5. 학생 프로젝트 확장 (추가 모델 훈련 없음)

**기본 과제**

1. 주어진 질문 8개에서 문자열 검색과 의미 검색의 결과를 비교한다. 어떤 오류가 발생하는지 최소 2개 설명한다.
2. 학생이 직접 질문을 **5개** 더 작성한다. 질문마다 관련 문서 번호를 스스로 정해 **Top-3 검색 결과**를 검토한다.
3. 검색된 문서를 근거로 **답변을 3개 이상 생성**하고, 문서에 적힌 내용과 일치하는지 직접 확인한다.
4. 문서에 없는 질문을 만들어 모델이 어떻게 답하는지 관찰한다. **근거가 있는 답변과 추측성 답변을 구분**한다.

**선택 확장**

- `TOP_K=1`, `3`, `5`를 비교한다. 문서가 많다고 반드시 답변이 좋아지지 않는 이유를 생각해 본다.
- 관심 주제의 공개된 문서나 자신이 작성한 짧은 설명문 **20~30개**로 기본 자료를 대체한다. 원본 출처와 이용 조건을 기록한다.
- 교육용 데이터 25개를 유지한 채 질문의 표현(동의어, 긴 문장, 띄어쓰기)을 바꾸어 검색의 안정성을 분석한다.

**보고서 작성 예시**

| 질문 | 기준 문서 | 검색된 문서 | 답변이 근거와 일치? | 오류·개선 아이디어 |
|---|---|---|---|---|
| 예: 장비는 언제 확인하나요? | D05 | 실행 결과 기록 | 직접 확인 | 질문 표현 변경 등 |

**제출 시 주의:** 모델이 생성한 답변을 사실이라고 간주하지 않는다. 인용된 문서를 직접 열어 근거를 확인해야 한다.


## 정리: P4 Theory와 연결

- **Encoder-only:** 문서와 질문을 의미를 담은 벡터로 표현하여 검색에 활용
- **Encoder–Decoder:** P4 Lab에서 영어 문장을 스페인어 문장으로 번역하는 데 사용
- **Decoder-only:** 검색된 문서를 프롬프트로 받아 다음 토큰을 반복적으로 생성하여 답변 작성

**이번 프로젝트는 Encoder-only + Decoder-only의 협업**을 보여준다. 다만 이를 통해 모델이 지식을 완벽히 검증하거나, 문서 바깥의 사실이 참이라고 확인하는 것은 아니다. **검색 품질과 답변의 근거 적합성을 모두 평가**해야 한다.

**참고 모델 링크**

- [다국어 문장 임베딩 모델](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)
- [소형 Decoder-only 생성 모델](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
